# LEAN: realistic limit fills in one line

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/sarptandoven/queuetruth/blob/main/examples/notebooks/08_lean.ipynb)

LEAN's default fill model fills a limit order as soon as a bar trades at or through its price. queuetruth's
`QueueFillModel` keeps LEAN's own limit logic and accepts each such fill with the probability the atlas touch
tables give for the order's tier, side and bar length; `qt.check` reads LEAN's order events and says how wrong the
run was:

```python
security.set_fill_model(qt.lean.QueueFillModel(tier="T1"))
```

Data: queuetruth's **synthetic** dataset and the repository's **synthetic** atlas (invented numbers). The numbers
only show the workflow; they say nothing about a real venue.

In [1]:
# Setup: on Colab (or anywhere queuetruth is missing) this installs it from PyPI; elsewhere it does nothing.
import importlib
import importlib.util
import subprocess
import sys

if importlib.util.find_spec("queuetruth") is None:
    try:
        from IPython import get_ipython

        shell = get_ipython()
    except ImportError:
        shell = None
    if shell is not None:  # %pip lets Colab offer a restart when an already-imported package was upgraded
        shell.run_line_magic("pip", "install -q queuetruth pandas matplotlib")
    else:
        subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", "queuetruth", "pandas", "matplotlib"])
    importlib.invalidate_caches()
if importlib.util.find_spec("queuetruth") is None or importlib.util.find_spec("queuetruth.frames") is None:
    raise RuntimeError(
        "this notebook needs queuetruth.frames (first released after queuetruth 0.1.1): pip install -U queuetruth"
    )

In [2]:
import importlib.util
import json
from pathlib import Path

import pandas as pd
from IPython.display import display

import queuetruth as qt
from queuetruth.backtesters.touchfill import TouchFill

pd.set_option("display.width", 140)
WORK = Path("qt-notebooks") / "08_lean"
WORK.mkdir(parents=True, exist_ok=True)

# queuetruth's synthetic order-level dataset: two futures (QTAU6, QTBU6), two days, invented prices.
DATA = WORK / "synth"
if not (DATA / "manifest.json").exists():
    qt.fixtures_synth(str(DATA))

# The repository's synthetic atlas (invented numbers, no market data): the checkout's copy, else the one on GitHub.
# Leave `source=` out to use the published atlas (it asks before downloading anything).
_local = Path(qt.__file__).resolve().parents[2] / "tests" / "fixtures" / "atlas_synth"
ATLAS = (
    str(_local)
    if (_local / "atlas" / "v1" / "manifest.json").exists()
    else "https://raw.githubusercontent.com/sarptandoven/queuetruth/main/tests/fixtures/atlas_synth/atlas/v1/"
)

# Twelve resting limit orders on the synthetic futures, as a bar backtest reports them: placed at `placed` (UTC),
# and filled 5 s later when `filled` (the touch rule), else cancelled 30 s later.
ORDERS = pd.DataFrame(
    [
        ("O-1", "QTBU6", "sell", 1, 5008.50, "2026-06-16 14:38:42", True),
        ("O-2", "QTAU6", "buy", 2, 4999.75, "2026-06-16 14:38:05", False),
        ("O-3", "QTBU6", "sell", 3, 5010.25, "2026-06-15 17:09:27", True),
        ("O-4", "QTAU6", "buy", 1, 4999.50, "2026-06-16 17:32:15", False),
        ("O-5", "QTAU6", "sell", 2, 5002.75, "2026-06-15 15:39:13", True),
        ("O-6", "QTAU6", "buy", 3, 5004.00, "2026-06-15 19:04:05", False),
        ("O-7", "QTBU6", "buy", 1, 5008.75, "2026-06-15 15:36:19", True),
        ("O-8", "QTBU6", "sell", 2, 5008.50, "2026-06-16 16:58:59", False),
        ("O-9", "QTAU6", "buy", 3, 4999.50, "2026-06-16 14:39:20", True),
        ("O-10", "QTAU6", "buy", 1, 4999.50, "2026-06-16 17:25:05", False),
        ("O-11", "QTBU6", "buy", 2, 5008.25, "2026-06-16 16:12:29", True),
        ("O-12", "QTAU6", "buy", 3, 5001.25, "2026-06-15 15:05:05", False),
    ],
    columns=["order", "symbol", "side", "qty", "price", "placed", "filled"],
)
ORDERS["placed"] = pd.to_datetime(ORDERS["placed"], utc=True)
ORDERS["done"] = ORDERS["placed"] + pd.to_timedelta(ORDERS["filled"].map({True: 5, False: 30}), unit="s")


def have(*modules, hint=None):
    """True when every module is importable; else prints which are missing and how to get them."""
    missing = [m for m in modules if importlib.util.find_spec(m) is None]
    if missing:
        print(f"skipped: {', '.join(missing)} not installed here ({hint or 'pip install ' + ' '.join(missing)})")
    return not missing


print("queuetruth", qt.__version__, "| SYNTHETIC data and a SYNTHETIC atlas: the numbers show the workflow only")

queuetruth 0.3.0 | SYNTHETIC data and a SYNTHETIC atlas: the numbers show the workflow only


## 1. What the fill model draws from

The touch tables give `P(real | touch)`: the probability that a resting limit order the bar touched really filled
in the queue, by tier, side and bar length (here the synthetic atlas; without `atlas=` the published one, after
asking).

In [3]:
touch = TouchFill(qt.atlas.touch_tables(ATLAS, yes=True), tier="T1")
display(
    pd.DataFrame(
        [{"bar": b, "side": s, "P(real | touch)": touch.probability(b, s)} for b in touch.bars for s in ("buy", "sell")]
    )
)

,bar,side,P(real | touch)
0,1h,buy,0.65
1,1h,sell,0.67
2,1m,buy,0.35
3,1m,sell,0.37
4,5m,buy,0.50
5,5m,sell,0.52


## 2. The one line

In a LEAN Python algorithm, after adding the security:

```python
from AlgorithmImports import *
import queuetruth as qt

class MyAlgorithm(QCAlgorithm):
    def initialize(self):
        security = self.add_equity("SPY", Resolution.MINUTE)
        security.set_fill_model(qt.lean.QueueFillModel(tier="T1"))
```

LEAN runs algorithms in its own engine, so the model is made only where `AlgorithmImports` exists:

In [4]:
if have("AlgorithmImports", hint="LEAN runs algorithms in its own engine: use the lean CLI"):
    model = qt.lean.QueueFillModel(tier="T1", atlas=ATLAS, yes=True)
    print(type(model).__name__, model.summary())

skipped: AlgorithmImports not installed here (LEAN runs algorithms in its own engine: use the lean CLI)


## 3. How wrong was the run?

`qt.check` reads LEAN's order events (the `*-order-events.json` of a backtest). Here they are written for twelve
orders on the synthetic futures; use your own. Without order-level data the answer is the atlas estimate:

In [5]:
events = []
for k, o in enumerate(ORDERS.itertuples(), start=1):
    sign = 1 if o.side == "buy" else -1
    base = {"algorithmId": "synthetic", "orderId": k, "symbol": o.symbol, "symbolValue": o.symbol,
            "symbolPermtick": o.symbol, "fillPriceCurrency": "USD", "direction": o.side, "orderType": "limit",
            "isAssignment": False, "quantity": sign * o.qty, "limitPrice": o.price}  # fmt: skip
    events.append({**base, "id": f"synthetic-{k}-1", "orderEventId": 1, "time": o.placed.timestamp(),
                   "status": "submitted", "fillPrice": 0.0, "fillQuantity": 0.0})  # fmt: skip
    if o.filled:
        done = {"status": "filled", "fillPrice": o.price, "fillQuantity": sign * o.qty, "orderFeeAmount": 0.0}
    else:
        done = {"status": "canceled", "fillPrice": 0.0, "fillQuantity": 0.0}
    events.append({**base, **done, "id": f"synthetic-{k}-2", "orderEventId": 2, "time": o.done.timestamp()})
path = WORK / "lean-order-events.json"
path.write_text(json.dumps(events))

estimate = qt.check(path, bar="5m", tier="T1", atlas=ATLAS, yes=True, out=WORK / "estimate")
display(estimate)

tier,tier_name,side,bar,orders,backtest_filled,p_real_given_touch,p_real_given_touch_ci_lo,p_real_given_touch_ci_hi,expected_real_fills,pnl_overstated_ticks_per_order,pnl_overstated_ticks_per_order_ci_lo,pnl_overstated_ticks_per_order_ci_hi,atlas_orders,atlas_dates,atlas_symbols
T1_top100,the 100 most traded symbols,buy,5m,8,3,0.5,0.45,0.55,1.5,0.4,0.3,0.5,4000,3,40
T1_top100,the 100 most traded symbols,sell,5m,4,3,0.52,0.47,0.57,1.56,0.4,0.3,0.5,4000,3,40


With order-level data of the same days, each order is replayed in the exact queue instead:

In [6]:
res = qt.check(path, data=DATA, horizon="1s", window="10s", out=WORK / "check")
display(res)

order,symbol,side,price,qty,entry,entry_local,backtest_filled,backtest_fill_after_s,real_filled,real_fill_after_s,queue_ahead,outcome,pnl_gap
sim:lean:backtest:1,QTBU6,sell,5008.5,1,2026-06-16T14:38:42.000+00:00,2026-06-16 14:38:42.000 UTC,True,5,True,2.497,1,both,0
sim:lean:backtest:10,QTAU6,buy,4999.5,1,2026-06-16T17:25:05.000+00:00,2026-06-16 17:25:05.000 UTC,False,,False,,12,neither,0
sim:lean:backtest:11,QTBU6,buy,5008.25,2,2026-06-16T16:12:29.000+00:00,2026-06-16 16:12:29.000 UTC,True,5,False,,0,phantom,25
sim:lean:backtest:12,QTAU6,buy,5001.25,3,2026-06-15T15:05:05.000+00:00,2026-06-15 15:05:05.000 UTC,False,,True,0.901,17,real_only,0
sim:lean:backtest:2,QTAU6,buy,4999.75,2,2026-06-16T14:38:05.000+00:00,2026-06-16 14:38:05.000 UTC,False,,False,,8,neither,0
sim:lean:backtest:3,QTBU6,sell,5010.25,3,2026-06-15T17:09:27.000+00:00,2026-06-15 17:09:27.000 UTC,True,5,False,,49,phantom,37.5
sim:lean:backtest:4,QTAU6,buy,4999.5,1,2026-06-16T17:32:15.000+00:00,2026-06-16 17:32:15.000 UTC,False,,True,0.371,1,real_only,6.25
sim:lean:backtest:5,QTAU6,sell,5002.75,2,2026-06-15T15:39:13.000+00:00,2026-06-15 15:39:13.000 UTC,True,5,False,,52,phantom,25
sim:lean:backtest:6,QTAU6,buy,5004,3,2026-06-15T19:04:05.000+00:00,2026-06-15 19:04:05.000 UTC,False,,False,,5,neither,0
sim:lean:backtest:7,QTBU6,buy,5008.75,1,2026-06-15T15:36:19.000+00:00,2026-06-15 15:36:19.000 UTC,True,5,False,,22,phantom,6.25


In [7]:
display(res.table()[["symbol", "side", "price", "entry_local", "backtest_filled", "real_filled", "outcome"]])

,symbol,side,price,entry_local,backtest_filled,real_filled,outcome
0,QTBU6,sell,5008.5,2026-06-16 14:38:42.000 UTC,True,True,both
1,QTAU6,buy,4999.5,2026-06-16 17:25:05.000 UTC,False,False,neither
2,QTBU6,buy,5008.25,2026-06-16 16:12:29.000 UTC,True,False,phantom
3,QTAU6,buy,5001.25,2026-06-15 15:05:05.000 UTC,False,True,real_only
4,QTAU6,buy,4999.75,2026-06-16 14:38:05.000 UTC,False,False,neither
5,QTBU6,sell,5010.25,2026-06-15 17:09:27.000 UTC,True,False,phantom
6,QTAU6,buy,4999.5,2026-06-16 17:32:15.000 UTC,False,True,real_only
7,QTAU6,sell,5002.75,2026-06-15 15:39:13.000 UTC,True,False,phantom
8,QTAU6,buy,5004,2026-06-15 19:04:05.000 UTC,False,False,neither
9,QTBU6,buy,5008.75,2026-06-15 15:36:19.000 UTC,True,False,phantom
